In [ ]:
# ==============================================================================
# DRUG FIGHTER v3 - MOLECULAR WEAPON & ANTHROPOMORPHIC RIG ARENA
# ==============================================================================
import os
import sys
import time
import json
import re
import subprocess
import threading

print("📦 [1/4] Đang cài đặt thư viện RDKit, Flask và Cloudflare Tunnel...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit", "flask", "flask-cors"], check=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from flask import Flask, jsonify, request, render_template_string
from flask_cors import CORS
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Draw import rdMolDraw2D

print("🧬 [2/4] Khởi tạo Bộ xử lý Vũ khí Phân tử RDKit...")

app = Flask(__name__)
CORS(app)

def process_smiles_weapon(smiles_str):
    try:
        smiles_str = smiles_str.strip()
        mol = Chem.MolFromSmiles(smiles_str)
        if mol is None:
            return {"success": False, "error": "Chuỗi SMILES không hợp lệ!"}

        # 1. Vẽ cấu trúc 2D SVG hiển thị ở khung Avatar
        drawer = rdMolDraw2D.MolDraw2DSVG(160, 160)
        opts = drawer.drawOptions()
        opts.clearBackground = False
        opts.setSymbolColour((1, 1, 1))
        opts.bondLineWidth = 2.5
        rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol)
        drawer.FinishDrawing()
        svg_2d = drawer.GetDrawingText()

        # 2. Sinh tọa độ 3D và tối ưu trường lực MMFF94 cho vũ khí
        mol_3d = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        if AllChem.EmbedMolecule(mol_3d, params) < 0:
            AllChem.EmbedMolecule(mol_3d, useRandomCoords=True)
        try:
            AllChem.MMFFOptimizeMolecule(mol_3d, maxIters=300)
        except Exception:
            pass

        # Lấy khung nguyên tử nặng để dựng Ball-and-Stick cầm tay sắc nét
        mol_heavy = Chem.RemoveHs(mol_3d)
        conf = mol_heavy.GetConformer()

        xs, ys, zs = [], [], []
        raw_atoms = []
        for atom in mol_heavy.GetAtoms():
            idx = atom.GetIdx()
            pos = conf.GetAtomPosition(idx)
            xs.append(pos.x)
            ys.append(pos.y)
            zs.append(pos.z)
            raw_atoms.append((idx, atom.GetSymbol()))

        cx = sum(xs) / len(xs)
        cy = sum(ys) / len(ys)
        cz = sum(zs) / len(zs)
        max_span = max(max(xs) - min(xs), max(ys) - min(ys), max(zs) - min(zs), 0.1)
        scale = 1.0 / max_span  # Tỉ lệ vừa vặn với kích cỡ bàn tay

        atoms = []
        for i, (idx, symbol) in enumerate(raw_atoms):
            atoms.append({
                "id": idx,
                "elem": symbol,
                "x": float((xs[i] - cx) * scale),
                "y": float((ys[i] - cy) * scale),
                "z": float((zs[i] - cz) * scale)
            })

        bonds = []
        for bond in mol_heavy.GetBonds():
            bonds.append({
                "u": bond.GetBeginAtomIdx(),
                "v": bond.GetEndAtomIdx(),
                "order": bond.GetBondTypeAsDouble()
            })

        mw = float(Descriptors.MolWt(mol))
        logp = float(Descriptors.MolLogP(mol))
        hbd = int(Descriptors.NumHDonors(mol))
        hba = int(Descriptors.NumHAcceptors(mol))
        formula = Chem.rdMolDescriptors.CalcMolFormula(mol)

        return {
            "success": True,
            "smiles": smiles_str,
            "formula": formula,
            "svg_2d": svg_2d,
            "atoms": atoms,
            "bonds": bonds,
            "stats": {
                "mw": round(mw, 1),
                "logp": round(logp, 2),
                "hp": int(100 + min(mw * 0.15, 80)),
                "atk": int(12 + max(logp * 3.5, 4)),
                "def": int(min(hbd + hba, 14) * 2)
            }
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.route('/api/smiles', methods=['POST'])
def api_smiles():
    data = request.json or {}
    smiles = data.get('smiles', 'CC(=O)Oc1ccccc1C(=O)O')
    res = process_smiles_weapon(smiles)
    if not res.get("success"):
        return jsonify(res), 400
    return jsonify(res)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>DRUG FIGHTER - Molecular Arena</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Chakra+Petch:wght@400;600;700;800&family=JetBrains+Mono:wght@500;700;800&display=swap');

  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: #060b13;
    color: #fff;
    font-family: 'Chakra Petch', sans-serif;
    overflow: hidden;
    user-select: none;
    width: 100vw;
    height: 100vh;
  }

  #glCanvas {
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    z-index: 1;
  }

  #bgOverlay {
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    z-index: 2;
    pointer-events: none;
  }

  #uiOverlay {
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    z-index: 10;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    pointer-events: none;
  }

  /* TOP HUD */
  .hud-top {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    padding: 16px 28px;
    width: 100%;
  }

  .fighter-panel {
    display: flex;
    gap: 12px;
    align-items: flex-start;
    width: 38%;
  }
  .fighter-panel.p2 { flex-direction: row-reverse; }

  .avatar-box {
    width: 82px;
    height: 82px;
    background: #235d3a;
    border: 2px solid #368a57;
    box-shadow: 0 0 14px rgba(35, 93, 58, 0.7);
    display: flex;
    align-items: center;
    justify-content: center;
    overflow: hidden;
  }
  .avatar-box svg { width: 100%; height: 100%; }

  .bar-meta { flex: 1; }
  .player-label-row {
    display: flex;
    justify-content: space-between;
    font-size: 13px;
    font-weight: 700;
    margin-bottom: 4px;
    letter-spacing: 1px;
    color: #e2e8f0;
  }
  .p2 .player-label-row { flex-direction: row-reverse; }
  .plddt-tag { font-family: 'JetBrains Mono', monospace; font-size: 11px; color: #94a3b8; }
  .dots { letter-spacing: 3px; color: #64748b; }

  .hp-track {
    width: 100%;
    height: 16px;
    background: #0f172a;
    border: 1px solid #334155;
    position: relative;
    overflow: hidden;
  }
  .hp-fill {
    height: 100%;
    width: 100%;
    background: linear-gradient(90deg, #38bdf8, #ffffff);
    box-shadow: 0 0 10px #38bdf8;
    transition: width 0.15s ease-out;
  }
  .p2 .hp-fill {
    background: linear-gradient(90deg, #ffffff, #38bdf8);
    float: right;
  }

  .timer-center {
    display: flex;
    flex-direction: column;
    align-items: center;
    margin-top: 4px;
  }
  .timer-val {
    font-family: 'JetBrains Mono', monospace;
    font-size: 42px;
    font-weight: 800;
    line-height: 1;
    color: #f8fafc;
    text-shadow: 0 0 12px rgba(255, 255, 255, 0.4);
  }
  .round-tag { font-size: 11px; letter-spacing: 2px; color: #94a3b8; margin-top: 2px; }

  /* CENTER SETTINGS MODAL */
  .center-modal {
    position: absolute;
    top: 44%;
    left: 50%;
    transform: translate(-50%, -50%);
    background: rgba(15, 23, 42, 0.92);
    border: 1px solid #334155;
    backdrop-filter: blur(8px);
    border-radius: 8px;
    padding: 18px;
    text-align: center;
    width: 320px;
    pointer-events: auto;
    box-shadow: 0 12px 32px rgba(0,0,0,0.7);
    z-index: 20;
  }
  .btn-start {
    background: #f1f5f9;
    color: #0f172a;
    font-family: 'Chakra Petch', sans-serif;
    font-weight: 800;
    font-size: 16px;
    border: none;
    padding: 8px 36px;
    border-radius: 4px;
    cursor: pointer;
    margin-bottom: 14px;
    letter-spacing: 2px;
    transition: all 0.2s;
  }
  .btn-start:hover {
    background: #38bdf8;
    color: #fff;
    box-shadow: 0 0 14px #38bdf8;
  }
  .menu-group { margin-bottom: 10px; }
  .menu-group-label {
    font-size: 10px;
    letter-spacing: 2px;
    color: #64748b;
    margin-bottom: 4px;
    text-transform: uppercase;
  }
  .pill-group {
    display: flex;
    justify-content: center;
    background: #0b1120;
    border: 1px solid #1e293b;
    border-radius: 4px;
    overflow: hidden;
  }
  .pill-btn {
    flex: 1;
    background: transparent;
    border: none;
    color: #94a3b8;
    font-family: 'Chakra Petch', sans-serif;
    font-size: 11px;
    padding: 6px 8px;
    cursor: pointer;
  }
  .pill-btn.active {
    background: #f8fafc;
    color: #0f172a;
    font-weight: 700;
  }

  /* CHARACTER SELECTION PILLS */
  .char-select-bar {
    position: absolute;
    bottom: 110px;
    width: 100%;
    display: flex;
    justify-content: space-around;
    padding: 0 80px;
    pointer-events: auto;
  }
  .char-pills {
    display: flex;
    background: rgba(15, 23, 42, 0.85);
    border: 1px solid #334155;
    border-radius: 20px;
    padding: 3px;
    gap: 4px;
  }
  .char-pill-btn {
    background: transparent;
    color: #94a3b8;
    border: none;
    padding: 5px 14px;
    border-radius: 16px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
    cursor: pointer;
    font-family: 'Chakra Petch', sans-serif;
    transition: 0.2s;
  }
  .char-pill-btn.active {
    background: #38bdf8;
    color: #04101c;
  }

  /* CONTROLS HUD (BOTTOM) */
  .hud-bottom {
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    padding: 14px 28px;
    width: 100%;
    pointer-events: auto;
  }
  .keybinds-cluster { display: flex; flex-direction: column; gap: 4px; }
  .keys-row { display: flex; gap: 4px; }
  .keycap {
    width: 32px;
    height: 32px;
    background: #0f172a;
    border: 1px solid #334155;
    border-radius: 4px;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    font-weight: 700;
    color: #f8fafc;
  }
  .keycap span { font-size: 7px; color: #64748b; margin-top: -2px; }
  .footer-credits { font-size: 10px; color: #475569; letter-spacing: 1px; }

  /* SMILES CUSTOM MODAL */
  #smilesModal {
    display: none;
    position: absolute;
    top: 50%; left: 50%;
    transform: translate(-50%, -50%);
    background: #0a1120;
    border: 1px solid #38bdf8;
    box-shadow: 0 0 35px rgba(56, 189, 248, 0.4);
    border-radius: 8px;
    padding: 22px;
    width: 480px;
    z-index: 100;
    pointer-events: auto;
  }
  #smilesModal h3 {
    font-size: 16px;
    color: #38bdf8;
    margin-bottom: 12px;
    letter-spacing: 1px;
  }
  #smilesInput {
    width: 100%;
    background: #050a14;
    border: 1px solid #334155;
    color: #fff;
    padding: 10px 12px;
    border-radius: 4px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 12px;
    margin-bottom: 12px;
    outline: none;
  }
  #smilesInput:focus { border-color: #38bdf8; }
  .quick-pills { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 16px; }
  .quick-tag {
    background: #1e293b;
    color: #94a3b8;
    border: 1px solid #334155;
    padding: 3px 8px;
    border-radius: 4px;
    font-size: 10px;
    cursor: pointer;
  }
  .quick-tag:hover { background: #334155; color: #fff; }
  .modal-actions { display: flex; justify-content: flex-end; gap: 8px; }
  .modal-btn {
    padding: 7px 16px;
    border-radius: 4px;
    border: none;
    font-family: 'Chakra Petch', sans-serif;
    font-size: 12px;
    cursor: pointer;
  }
  .modal-btn.primary { background: #38bdf8; color: #04101c; font-weight: 700; }
  .modal-btn.secondary { background: #334155; color: #fff; }

  /* K.O / END GAME MODAL */
  #gameOverModal {
    display: none;
    position: absolute;
    top: 50%; left: 50%;
    transform: translate(-50%, -50%);
    background: rgba(10, 17, 32, 0.95);
    border: 2px solid #38bdf8;
    box-shadow: 0 0 50px rgba(56, 189, 248, 0.5);
    border-radius: 12px;
    padding: 30px;
    text-align: center;
    width: 380px;
    z-index: 150;
    pointer-events: auto;
  }
  .ko-title {
    font-family: 'JetBrains Mono', monospace;
    font-size: 32px;
    font-weight: 800;
    color: #ef4444;
    letter-spacing: 3px;
    margin-bottom: 8px;
    text-shadow: 0 0 20px rgba(239, 68, 68, 0.8);
  }
  .ko-winner {
    font-size: 18px;
    font-weight: 700;
    color: #38bdf8;
    margin-bottom: 22px;
    letter-spacing: 1px;
  }
  .ko-actions {
    display: flex;
    flex-direction: column;
    gap: 10px;
  }
  .btn-ko {
    padding: 10px;
    border-radius: 6px;
    border: none;
    font-family: 'Chakra Petch', sans-serif;
    font-size: 14px;
    font-weight: 700;
    cursor: pointer;
    transition: 0.2s;
  }
  .btn-ko.rematch { background: #38bdf8; color: #04101c; }
  .btn-ko.rematch:hover { background: #7dd3fc; }
  .btn-ko.menu { background: #1e293b; color: #f8fafc; border: 1px solid #334155; }
  .btn-ko.menu:hover { background: #334155; }
</style>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
</head>
<body>

<canvas id="glCanvas"></canvas>
<canvas id="bgOverlay"></canvas>

<div id="uiOverlay">
  <!-- Top HUD -->
  <div class="hud-top">
    <!-- P1 -->
    <div class="fighter-panel p1">
      <div class="avatar-box" id="p1-avatar"></div>
      <div class="bar-meta">
        <div class="player-label-row">
          <span id="p1-title">P1 - ASPIRIN</span>
          <span class="plddt-tag">pLDDT 100 &nbsp;<span class="dots">● ●</span></span>
        </div>
        <div class="hp-track">
          <div class="hp-fill" id="p1-hp"></div>
        </div>
      </div>
    </div>

    <!-- Timer -->
    <div class="timer-center">
      <div class="timer-val" id="timerVal">99</div>
      <div class="round-tag">ROUND 01</div>
    </div>

    <!-- P2 -->
    <div class="fighter-panel p2">
      <div class="avatar-box" id="p2-avatar"></div>
      <div class="bar-meta">
        <div class="player-label-row">
          <span id="p2-title">P2 - PARACETAMOL</span>
          <span class="plddt-tag"><span class="dots">● ●</span> &nbsp;pLDDT 100</span>
        </div>
        <div class="hp-track">
          <div class="hp-fill" id="p2-hp"></div>
        </div>
      </div>
    </div>
  </div>

  <!-- Center Menu -->
  <div class="center-modal" id="centerMenu">
    <button class="btn-start" id="btnStartGame">START</button>
    <div class="menu-group">
      <div class="menu-group-label">Players</div>
      <div class="pill-group">
        <button class="pill-btn active" id="btnMode1P">1 PLAYER</button>
        <button class="pill-btn" id="btnMode2P">2 PLAYERS</button>
        <button class="pill-btn">REMOTE</button>
      </div>
    </div>
    <div class="menu-group">
      <div class="menu-group-label">CPU</div>
      <div class="pill-group">
        <button class="pill-btn">EASY</button>
        <button class="pill-btn active">NORMAL</button>
        <button class="pill-btn">HARD</button>
      </div>
    </div>
    <div class="menu-group">
      <div class="menu-group-label">Colour</div>
      <div class="pill-group">
        <button class="pill-btn active">pLDDT</button>
        <button class="pill-btn">RAINBOW</button>
        <button class="pill-btn">SSE</button>
      </div>
    </div>
  </div>

  <!-- Character selection pills -->
  <div class="char-select-bar" id="charSelectBar">
    <div class="char-pills" id="p1-pills">
      <button class="char-pill-btn active" onclick="loadPreset(1, 'Aspirin', this)">ASPIRIN</button>
      <button class="char-pill-btn" onclick="loadPreset(1, 'Caffeine', this)">CAFFEINE</button>
      <button class="char-pill-btn" onclick="openSmilesModal(1)">CUSTOM</button>
    </div>
    <div class="char-pills" id="p2-pills">
      <button class="char-pill-btn active" onclick="loadPreset(2, 'Paracetamol', this)">PARACETAMOL</button>
      <button class="char-pill-btn" onclick="loadPreset(2, 'Ibuprofen', this)">IBUPROFEN</button>
      <button class="char-pill-btn" onclick="openSmilesModal(2)">CUSTOM</button>
    </div>
  </div>

  <!-- Bottom Keybinds -->
  <div class="hud-bottom">
    <div class="keybinds-cluster">
      <div class="keys-row" style="margin-left: 36px;">
        <div class="keycap">W<span>JUMP</span></div>
      </div>
      <div class="keys-row">
        <div class="keycap">A<span>LEFT</span></div>
        <div class="keycap">S<span>CROUCH</span></div>
        <div class="keycap">D<span>RIGHT</span></div>
        <div class="keycap" style="margin-left: 10px;">F<span>PUNCH</span></div>
        <div class="keycap">G<span>KICK</span></div>
        <div class="keycap">H<span>BLOCK</span></div>
      </div>
    </div>
    <div class="footer-credits">
      DRUG FIGHTER • Molecular Rig Inspired by Nanokid & sokrypton • Powered by RDKit & Three.js
    </div>
  </div>
</div>

<!-- SMILES Custom Dialog -->
<div id="smilesModal">
  <h3 id="smilesModalTitle">ENTER CUSTOM SMILES</h3>
  <input type="text" id="smilesInput" placeholder="Nhập SMILES (VD: CC(=O)Oc1ccccc1C(=O)O)">
  <div style="font-size:11px; color:#64748b; margin-bottom:6px;">GỢI Ý THUỐC LÀM VŨ KHÍ:</div>
  <div class="quick-pills">
    <span class="quick-tag" onclick="setSmiles('CN1CCC23C4C1CC5=C2C(=C(C=C5)O)OC3C(C=C4)O')">Morphine</span>
    <span class="quick-tag" onclick="setSmiles('CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C')">Penicillin</span>
    <span class="quick-tag" onclick="setSmiles('C1=CC(=C(C=C1CCN)O)O')">Dopamine</span>
    <span class="quick-tag" onclick="setSmiles('CN1CCCC1c2cccnc2')">Nicotine</span>
  </div>
  <div class="modal-actions">
    <button class="modal-btn secondary" onclick="closeSmilesModal()">CANCEL</button>
    <button class="modal-btn primary" id="btnSubmitSmiles">EQUIP WEAPON</button>
  </div>
</div>

<!-- Game Over Modal -->
<div id="gameOverModal">
  <div class="ko-title">KNOCKOUT!</div>
  <div class="ko-winner" id="koWinnerMsg">PLAYER 1 CHIẾN THẮNG</div>
  <div class="ko-actions">
    <button class="btn-ko rematch" onclick="rematchGame()">ĐẤU LẠI (REMATCH)</button>
    <button class="btn-ko menu" onclick="returnToMainMenu()">QUAY LẠI MÀN HÌNH CHÍNH</button>
  </div>
</div>

<script>
const PRESETS = {
  Aspirin: "CC(=O)Oc1ccccc1C(=O)O",
  Caffeine: "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
  Paracetamol: "CC(=O)Nc1ccc(O)cc1",
  Ibuprofen: "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
};

// AUDIO SYNTHESIZER
const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
function playAudio(type) {
  if (audioCtx.state === 'suspended') audioCtx.resume();
  const osc = audioCtx.createOscillator();
  const gain = audioCtx.createGain();
  osc.connect(gain);
  gain.connect(audioCtx.destination);
  const now = audioCtx.currentTime;
  if (type === 'hit') {
    osc.type = 'triangle';
    osc.frequency.setValueAtTime(150, now);
    osc.frequency.exponentialRampToValueAtTime(30, now + 0.12);
    gain.gain.setValueAtTime(0.35, now);
    gain.gain.linearRampToValueAtTime(0.01, now + 0.12);
    osc.start(now); osc.stop(now + 0.12);
  } else if (type === 'swing') {
    osc.type = 'sine';
    osc.frequency.setValueAtTime(320, now);
    osc.frequency.linearRampToValueAtTime(120, now + 0.09);
    gain.gain.setValueAtTime(0.15, now);
    gain.gain.linearRampToValueAtTime(0.01, now + 0.09);
    osc.start(now); osc.stop(now + 0.09);
  } else if (type === 'block') {
    osc.type = 'square';
    osc.frequency.setValueAtTime(450, now);
    osc.frequency.linearRampToValueAtTime(280, now + 0.08);
    gain.gain.setValueAtTime(0.15, now);
    gain.gain.linearRampToValueAtTime(0.01, now + 0.08);
    osc.start(now); osc.stop(now + 0.08);
  }
}

// 2D BACKGROUND (Mitochondria & Lipid Bilayer)
const bgCanvas = document.getElementById('bgOverlay');
const bgCtx = bgCanvas.getContext('2d');
function resizeBg() {
  bgCanvas.width = window.innerWidth;
  bgCanvas.height = window.innerHeight;
  drawCellularBackground();
}
window.addEventListener('resize', resizeBg);

function drawMitochondrion(ctx, cx, cy, radX, radY, rot) {
  ctx.save();
  ctx.translate(cx, cy);
  ctx.rotate(rot);
  ctx.strokeStyle = 'rgba(180, 83, 60, 0.45)';
  ctx.lineWidth = 2.5;
  ctx.beginPath();
  ctx.ellipse(0, 0, radX, radY, 0, 0, Math.PI * 2);
  ctx.stroke();
  ctx.beginPath();
  ctx.ellipse(0, 0, radX - 8, radY - 6, 0, 0, Math.PI * 2);
  ctx.stroke();
  for (let i = -radX + 24; i < radX - 20; i += 22) {
    ctx.beginPath();
    ctx.moveTo(i, -radY + 14);
    ctx.quadraticCurveTo(i + 8, 0, i, radY - 14);
    ctx.stroke();
  }
  ctx.restore();
}

function drawLipidBilayer(ctx, w, h) {
  const y = h - 60;
  ctx.strokeStyle = '#38bdf8';
  ctx.fillStyle = '#0ea5e9';
  const spacing = 18;
  for (let x = 0; x < w; x += spacing) {
    ctx.beginPath();
    ctx.arc(x, y - 12, 4.5, 0, Math.PI * 2);
    ctx.fill();
    ctx.beginPath();
    ctx.arc(x, y + 12, 4.5, 0, Math.PI * 2);
    ctx.fill();
    ctx.beginPath();
    ctx.moveTo(x - 1.5, y - 8);
    ctx.lineTo(x - 2.5, y);
    ctx.lineTo(x - 1.5, y + 8);
    ctx.moveTo(x + 1.5, y - 8);
    ctx.lineTo(x + 2.5, y);
    ctx.lineTo(x + 1.5, y + 8);
    ctx.stroke();
  }
}

function drawCellularBackground() {
  bgCtx.clearRect(0, 0, bgCanvas.width, bgCanvas.height);
  const w = bgCanvas.width;
  const h = bgCanvas.height;
  drawMitochondrion(bgCtx, w * 0.12, h * 0.48, 110, 55, -0.2);
  drawMitochondrion(bgCtx, w * 0.88, h * 0.48, 110, 55, 0.2);
  bgCtx.strokeStyle = 'rgba(56, 189, 248, 0.2)';
  bgCtx.lineWidth = 1.5;
  bgCtx.beginPath();
  bgCtx.arc(w * 0.32, h * 0.35, 45, 0, Math.PI * 2);
  bgCtx.stroke();
  bgCtx.beginPath();
  bgCtx.arc(w * 0.68, h * 0.35, 45, 0, Math.PI * 2);
  bgCtx.stroke();
  drawLipidBilayer(bgCtx, w, h);
}

// THREE.JS SCENE SETUP
const canvas3D = document.getElementById('glCanvas');
const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(45, window.innerWidth / window.innerHeight, 0.1, 1000);
camera.position.set(0, 0, 16);

const renderer = new THREE.WebGLRenderer({ canvas: canvas3D, antialias: true, alpha: true });
renderer.setSize(window.innerWidth, window.innerHeight);
renderer.setPixelRatio(window.devicePixelRatio);

const hemiLight = new THREE.HemisphereLight(0xffffff, 0x080e1a, 1.4);
scene.add(hemiLight);
const dirLight = new THREE.DirectionalLight(0x38bdf8, 1.8);
dirLight.position.set(5, 12, 10);
scene.add(dirLight);

window.addEventListener('resize', () => {
  camera.aspect = window.innerWidth / window.innerHeight;
  camera.updateProjectionMatrix();
  renderer.setSize(window.innerWidth, window.innerHeight);
});

// CHEMICAL BOND 3D BUILDERS
const bondMat = new THREE.MeshPhongMaterial({ color: 0x38bdf8, shininess: 90 });
const atomMat = new THREE.MeshPhongMaterial({ color: 0xffffff, shininess: 100 });

function createCylinderBond(p1, p2, radius, mat = bondMat) {
  const dist = p1.distanceTo(p2);
  const geo = new THREE.CylinderGeometry(radius, radius, dist, 8);
  const mesh = new THREE.Mesh(geo, mat);
  const mid = new THREE.Vector3().addVectors(p1, p2).multiplyScalar(0.5);
  mesh.position.copy(mid);
  mesh.quaternion.setFromUnitVectors(new THREE.Vector3(0, 1, 0), new THREE.Vector3().subVectors(p2, p1).normalize());
  return mesh;
}

function createSingleBond(p1, p2, mat = bondMat) {
  return createCylinderBond(p1, p2, 0.045, mat);
}

function createDoubleBond(p1, p2, normalVec, mat = bondMat) {
  const group = new THREE.Group();
  const offset = normalVec.clone().normalize().multiplyScalar(0.06);
  group.add(createCylinderBond(p1.clone().add(offset), p2.clone().add(offset), 0.035, mat));
  group.add(createCylinderBond(p1.clone().sub(offset), p2.clone().sub(offset), 0.035, mat));
  return group;
}

function createTripleBond(p1, p2, normalVec, mat = bondMat) {
  const group = new THREE.Group();
  const offset = normalVec.clone().normalize().multiplyScalar(0.075);
  group.add(createCylinderBond(p1, p2, 0.032, mat)); // Center bond
  group.add(createCylinderBond(p1.clone().add(offset), p2.clone().add(offset), 0.032, mat));
  group.add(createCylinderBond(p1.clone().sub(offset), p2.clone().sub(offset), 0.032, mat));
  return group;
}

// ANTHROPOMORPHIC CHEMICAL FIGHTER RIG (DESIGN BASED ON USER IMAGE)
class AnthropomorphicMoleculeFighter {
  constructor(isP1, initialSmiles, name) {
    this.isP1 = isP1;
    this.name = name;
    this.x = isP1 ? -4.5 : 4.5;
    this.y = -1.2;
    this.baseY = -1.2;
    this.vx = 0;
    this.vy = 0;
    this.facing = isP1 ? 1 : -1;
    this.onGround = true;

    this.maxHp = 100;
    this.hp = 100;
    this.state = 'idle';
    this.stateTimer = 0;
    this.animTime = 0;

    this.group = new THREE.Group();
    scene.add(this.group);

    this.buildAnthropomorphicRig();
    this.loadSmilesWeapon(initialSmiles, name);
  }

  buildAnthropomorphicRig() {
    this.root = new THREE.Group();
    this.group.add(this.root);

    // Pelvis Hub (Connecting Alkyne Spine & Conjugated Diene Legs)
    this.pelvis = new THREE.Group();
    this.root.add(this.pelvis);

    // Spine Hub (Alkyne -C#C- Linker)
    this.spine = new THREE.Group();
    this.pelvis.add(this.spine);

    // Chest / Quaternary Central Hub Carbon
    this.chestHub = new THREE.Group();
    this.chestHub.position.y = 1.35;
    this.spine.add(this.chestHub);

    // Head: Phenyl Ring (Benzene)
    this.headGroup = new THREE.Group();
    this.headGroup.position.y = 0.55;
    this.chestHub.add(this.headGroup);
    this.buildBenzeneHead();

    // Arms: Alkyne Arms (-C#C-CH3)
    this.leftArm = new THREE.Group();
    this.leftArm.position.set(-0.35, 0.1, 0);
    this.chestHub.add(this.leftArm);

    this.rightArm = new THREE.Group();
    this.rightArm.position.set(0.35, 0.1, 0);
    this.chestHub.add(this.rightArm);

    // Build Arm Chemical Bones
    this.buildAlkyneArm(this.leftArm, -1);
    this.buildAlkyneArm(this.rightArm, 1);

    // Weapon Attachment Mount on Right Arm Hand (Front weapon)
    this.weaponMount = new THREE.Group();
    this.weaponMount.position.set(0.8, -0.6, 0.2);
    this.rightArm.add(this.weaponMount);

    // Build Spine Triple Bond (-C#C-)
    const pSpineBottom = new THREE.Vector3(0, 0, 0);
    const pSpineTop = new THREE.Vector3(0, 1.35, 0);
    this.spine.add(createTripleBond(pSpineBottom, pSpineTop, new THREE.Vector3(1, 0, 0)));

    // Legs: Conjugated Diene Legs (-CH=CH-CH=CH-CH3)
    this.leftLeg = new THREE.Group();
    this.leftLeg.position.set(-0.25, 0, 0);
    this.pelvis.add(this.leftLeg);
    this.leftShin = new THREE.Group();
    this.leftShin.position.set(-0.45, -0.7, 0);
    this.leftLeg.add(this.leftShin);

    this.rightLeg = new THREE.Group();
    this.rightLeg.position.set(0.25, 0, 0);
    this.pelvis.add(this.rightLeg);
    this.rightShin = new THREE.Group();
    this.rightShin.position.set(0.45, -0.7, 0);
    this.rightLeg.add(this.rightShin);

    this.buildDieneLegs();

    // Shield Forcefield
    const shieldGeo = new THREE.SphereGeometry(1.6, 16, 16);
    const shieldMat = new THREE.MeshBasicMaterial({ color: 0x38bdf8, wireframe: true, transparent: true, opacity: 0 });
    this.shieldMesh = new THREE.Mesh(shieldGeo, shieldMat);
    this.shieldMesh.position.y = 1.0;
    this.group.add(this.shieldMesh);
  }

  // 1. Dựng Vòng thơm Benzene trên đầu
  buildBenzeneHead() {
    const r = 0.5;
    const ringGroup = new THREE.Group();
    const pts = [];
    for (let i = 0; i < 6; i++) {
      const angle = (i * Math.PI) / 3 - Math.PI / 2;
      pts.push(new THREE.Vector3(Math.cos(angle) * r, Math.sin(angle) * r + 0.55, 0));
    }
    // 6 liên kết tạo thành vòng lục giác
    for (let i = 0; i < 6; i++) {
      const pA = pts[i];
      const pB = pts[(i + 1) % 6];
      if (i % 2 === 0) {
        ringGroup.add(createDoubleBond(pA, pB, new THREE.Vector3(pA.x + pB.x, pA.y + pB.y, 0).normalize()));
      } else {
        ringGroup.add(createSingleBond(pA, pB));
      }
      // Nút nguyên tử Carbon
      const cMesh = new THREE.Mesh(new THREE.SphereGeometry(0.06, 8, 8), atomMat);
      cMesh.position.copy(pA);
      ringGroup.add(cMesh);
    }
    // Liên kết cổ nối Benzene xuống tâm ngực
    ringGroup.add(createSingleBond(new THREE.Vector3(0, 0, 0), pts[0]));
    this.headGroup.add(ringGroup);
  }

  // 2. Dựng Cánh tay Alkyne (-C#C-CH3)
  buildAlkyneArm(parent, dir) {
    const pStart = new THREE.Vector3(0, 0, 0);
    const pAlkyneEnd = new THREE.Vector3(0.55 * dir, -0.4, 0);
    const pHand = new THREE.Vector3(0.85 * dir, -0.65, 0);

    // Đoạn liên kết ba Alkyne (-C#C-)
    parent.add(createTripleBond(pStart, pAlkyneEnd, new THREE.Vector3(-0.4, 0.55 * dir, 0).normalize()));
    // Đoạn liên kết đơn ra chót nhóm Methyl/Nắm đấm (-CH3)
    parent.add(createSingleBond(pAlkyneEnd, pHand));

    const fistMesh = new THREE.Mesh(new THREE.SphereGeometry(0.08, 8, 8), atomMat);
    fistMesh.position.copy(pHand);
    parent.add(fistMesh);
  }

  // 3. Dựng Chân liên hợp Diene (-CH=CH-CH=CH-CH3)
  buildDieneLegs() {
    // Chân trái: Thigh (C=C), Shin (C=C), Foot (Single bond -CH3)
    const pL1 = new THREE.Vector3(0, 0, 0);
    const pL2 = new THREE.Vector3(-0.45, -0.7, 0);
    this.leftLeg.add(createDoubleBond(pL1, pL2, new THREE.Vector3(0.7, -0.45, 0)));

    const pL3 = new THREE.Vector3(0, 0, 0);
    const pL4 = new THREE.Vector3(0, -0.75, 0);
    this.leftShin.add(createDoubleBond(pL3, pL4, new THREE.Vector3(1, 0, 0)));
    // Bàn chân
    this.leftShin.add(createSingleBond(pL4, new THREE.Vector3(-0.35, -0.75, 0)));

    // Chân phải: Thigh (C=C), Shin (C=C), Foot (Single bond -CH3)
    const pR1 = new THREE.Vector3(0, 0, 0);
    const pR2 = new THREE.Vector3(0.45, -0.7, 0);
    this.rightLeg.add(createDoubleBond(pR1, pR2, new THREE.Vector3(-0.7, 0.45, 0)));

    const pR3 = new THREE.Vector3(0, 0, 0);
    const pR4 = new THREE.Vector3(0, -0.75, 0);
    this.rightShin.add(createDoubleBond(pR3, pR4, new THREE.Vector3(1, 0, 0)));
    // Bàn chân
    this.rightShin.add(createSingleBond(pR4, new THREE.Vector3(0.35, -0.75, 0)));
  }

  // Nạp cấu trúc Vũ khí Phân tử 3D từ SMILES
  async loadSmilesWeapon(smiles, displayName) {
    try {
      const res = await fetch('/api/smiles', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ smiles })
      });
      const data = await res.json();
      if (!data.success) {
        alert("Lỗi RDKit: " + (data.error || "Không thể phân tích SMILES"));
        return;
      }

      this.data = data;
      this.maxHp = data.stats.hp;
      this.hp = this.maxHp;

      const pPrefix = this.isP1 ? 'p1' : 'p2';
      document.getElementById(`${pPrefix}-avatar`).innerHTML = data.svg_2d;
      document.getElementById(`${pPrefix}-title`).innerText = `${pPrefix.toUpperCase()} - ${displayName || data.formula}`;

      this.buildWeaponMesh(data.atoms, data.bonds);
    } catch (e) {
      console.error("Lỗi tải vũ khí phân tử:", e);
    }
  }

  // Dựng Ball-and-Stick cho Vũ khí Phân tử trên tay
  buildWeaponMesh(atoms, bonds) {
    while (this.weaponMount.children.length > 0) {
      const obj = this.weaponMount.children[0];
      if (obj.geometry) obj.geometry.dispose();
      this.weaponMount.remove(obj);
    }

    const atomSpheres = [];
    atoms.forEach(at => {
      let col = 0x38bdf8;
      if (at.elem === 'O') col = 0xef4444; // Oxy màu đỏ
      else if (at.elem === 'N') col = 0x3b82f6; // Nitơ màu xanh dương
      else if (at.elem === 'S') col = 0xeab308; // Lưu huỳnh màu vàng
      else if (at.elem === 'C') col = 0x0284c7; // Carbon màu xanh cyan

      const aMat = new THREE.MeshPhongMaterial({ color: col, shininess: 100 });
      const aGeo = new THREE.SphereGeometry(0.13, 10, 10);
      const aMesh = new THREE.Mesh(aGeo, aMat);
      aMesh.position.set(at.x * 1.3, at.y * 1.3, at.z * 1.3);
      this.weaponMount.add(aMesh);
      atomSpheres.push(aMesh);
    });

    bonds.forEach(b => {
      const p1 = atomSpheres[b.u].position;
      const p2 = atomSpheres[b.v].position;
      const dist = p1.distanceTo(p2);
      const bGeo = new THREE.CylinderGeometry(0.045, 0.045, dist, 8);
      const bMat = new THREE.MeshPhongMaterial({ color: 0xe0f2fe, shininess: 80 });
      const bMesh = new THREE.Mesh(bGeo, bMat);

      const mid = new THREE.Vector3().addVectors(p1, p2).multiplyScalar(0.5);
      bMesh.position.copy(mid);
      bMesh.quaternion.setFromUnitVectors(new THREE.Vector3(0, 1, 0), new THREE.Vector3().subVectors(p2, p1).normalize());
      this.weaponMount.add(bMesh);
    });

    // Hào quang năng lượng vũ khí
    const auraGeo = new THREE.SphereGeometry(0.85, 12, 12);
    const auraMat = new THREE.MeshBasicMaterial({ color: 0x38bdf8, wireframe: true, transparent: true, opacity: 0.35 });
    this.weaponMount.add(new THREE.Mesh(auraGeo, auraMat));
  }

  update(opponent) {
    this.animTime += 0.08;

    this.x += this.vx;
    this.y += this.vy;

    if (!this.onGround) {
      this.vy -= 0.025;
      if (this.y <= this.baseY) {
        this.y = this.baseY;
        this.vy = 0;
        this.onGround = true;
      }
    }

    this.x = Math.max(-6.5, Math.min(6.5, this.x));
    this.facing = (this.x < opponent.x) ? 1 : -1;
    this.group.position.set(this.x, this.y, 0);
    this.group.scale.x = this.facing;

    if (this.stateTimer > 0) {
      this.stateTimer--;
      if (this.stateTimer === 0) this.state = 'idle';
    }

    // Xoay nhẹ vũ khí phân tử tạo cảm giác tích năng lượng
    if (this.weaponMount) {
      this.weaponMount.rotation.y += 0.03;
      this.weaponMount.rotation.z += 0.015;
    }

    this.animateBones();
    this.shieldMesh.material.opacity = (this.state === 'block') ? 0.6 : 0;
  }

  animateBones() {
    const t = this.animTime;

    if (this.state === 'idle') {
      this.pelvis.position.y = Math.sin(t * 2) * 0.05;
      this.chestHub.rotation.z = 0.04;

      // Tay thủ quyền tự nhiên
      this.leftArm.rotation.z = 0.4 + Math.sin(t) * 0.06;
      this.rightArm.rotation.z = -0.5 + Math.sin(t) * 0.06;

      // Nhún gối
      this.leftLeg.rotation.z = 0.15;
      this.rightLeg.rotation.z = -0.15;
    } else if (this.state === 'crouch') {
      this.pelvis.position.y = -0.45;
      this.leftArm.rotation.z = 0.8;
      this.rightArm.rotation.z = -0.9;
    } else if (this.state === 'punch') {
      // Vung mạnh tay cầm vũ khí phân tử về phía trước
      this.rightArm.rotation.z = -1.6;
    } else if (this.state === 'kick') {
      // Tung chân đá thẳng
      this.rightLeg.rotation.z = 1.6;
    }
  }

  punch(opponent) {
    if (this.state !== 'idle') return;
    this.state = 'punch';
    this.stateTimer = 14;
    playAudio('swing');

    if (Math.abs(this.x - opponent.x) < 2.5) {
      const dmg = this.data ? this.data.stats.atk : 14;
      opponent.takeDamage(dmg, this.facing);
    }
  }

  kick(opponent) {
    if (this.state !== 'idle') return;
    this.state = 'kick';
    this.stateTimer = 18;
    playAudio('swing');

    if (Math.abs(this.x - opponent.x) < 2.8) {
      const dmg = this.data ? Math.floor(this.data.stats.atk * 1.3) : 18;
      opponent.takeDamage(dmg, this.facing * 1.5);
    }
  }

  takeDamage(amount, knockback) {
    if (this.state === 'block') {
      playAudio('block');
      this.hp = Math.max(0, this.hp - Math.floor(amount * 0.25));
      this.x += knockback * 0.2;
    } else {
      playAudio('hit');
      this.hp = Math.max(0, this.hp - amount);
      this.x += knockback * 0.5;
    }
  }
}

// KHỞI TẠO ĐẤU THỦ
let p1 = new AnthropomorphicMoleculeFighter(true, PRESETS.Aspirin, "ASPIRIN");
let p2 = new AnthropomorphicMoleculeFighter(false, PRESETS.Paracetamol, "PARACETAMOL");

let is2Player = false;
let gameRunning = false;
let timerSeconds = 99;

// BẮT SỰ KIỆN PHÍM
const keys = {};
window.addEventListener('keydown', (e) => { keys[e.key.toLowerCase()] = true; });
window.addEventListener('keyup', (e) => { keys[e.key.toLowerCase()] = false; });

document.getElementById('btnMode1P').addEventListener('click', function() {
  is2Player = false;
  this.classList.add('active');
  document.getElementById('btnMode2P').classList.remove('active');
});
document.getElementById('btnMode2P').addEventListener('click', function() {
  is2Player = true;
  this.classList.add('active');
  document.getElementById('btnMode1P').classList.remove('active');
});
document.getElementById('btnStartGame').addEventListener('click', () => {
  startGame();
});

function startGame() {
  document.getElementById('centerMenu').style.display = 'none';
  document.getElementById('charSelectBar').style.display = 'none';
  document.getElementById('gameOverModal').style.display = 'none';
  p1.hp = p1.maxHp;
  p2.hp = p2.maxHp;
  p1.x = -4.5;
  p2.x = 4.5;
  timerSeconds = 99;
  gameRunning = true;
}

function rematchGame() {
  startGame();
}

function returnToMainMenu() {
  document.getElementById('gameOverModal').style.display = 'none';
  document.getElementById('centerMenu').style.display = 'block';
  document.getElementById('charSelectBar').style.display = 'flex';
  gameRunning = false;
  p1.hp = p1.maxHp;
  p2.hp = p2.maxHp;
  p1.x = -4.5;
  p2.x = 4.5;
  timerSeconds = 99;
}

function triggerGameOver(winnerMsg) {
  gameRunning = false;
  document.getElementById('koWinnerMsg').innerText = winnerMsg;
  document.getElementById('gameOverModal').style.display = 'block';
}

function loadPreset(playerNum, drugName, btn) {
  const smiles = PRESETS[drugName];
  if (playerNum === 1) {
    p1.loadSmilesWeapon(smiles, drugName.toUpperCase());
    document.querySelectorAll('#p1-pills .char-pill-btn').forEach(b => b.classList.remove('active'));
  } else {
    p2.loadSmilesWeapon(smiles, drugName.toUpperCase());
    document.querySelectorAll('#p2-pills .char-pill-btn').forEach(b => b.classList.remove('active'));
  }
  if (btn) btn.classList.add('active');
}

let targetCustomPlayer = 1;
function openSmilesModal(playerNum) {
  targetCustomPlayer = playerNum;
  document.getElementById('smilesModalTitle').innerText = `NHẬP VŨ KHÍ SMILES CHO PLAYER ${playerNum}`;
  document.getElementById('smilesInput').value = '';
  document.getElementById('smilesModal').style.display = 'block';
}
function closeSmilesModal() {
  document.getElementById('smilesModal').style.display = 'none';
}
function setSmiles(s) {
  document.getElementById('smilesInput').value = s;
}

document.getElementById('btnSubmitSmiles').addEventListener('click', () => {
  const val = document.getElementById('smilesInput').value.trim();
  if (val) {
    if (targetCustomPlayer === 1) {
      p1.loadSmilesWeapon(val, "CUSTOM WEAPON");
      document.querySelectorAll('#p1-pills .char-pill-btn').forEach(b => b.classList.remove('active'));
      document.querySelectorAll('#p1-pills .char-pill-btn')[2].classList.add('active');
    } else {
      p2.loadSmilesWeapon(val, "CUSTOM WEAPON");
      document.querySelectorAll('#p2-pills .char-pill-btn').forEach(b => b.classList.remove('active'));
      document.querySelectorAll('#p2-pills .char-pill-btn')[2].classList.add('active');
    }
    closeSmilesModal();
  }
});

// AI BOT LOGIC CHO PLAYER 2
function updateBot(bot, player) {
  const dist = Math.abs(bot.x - player.x);
  if (dist > 2.4) {
    bot.vx = (bot.x > player.x) ? -0.06 : 0.06;
  } else {
    bot.vx = 0;
    if (Math.random() < 0.04) bot.punch(player);
    else if (Math.random() < 0.03) bot.kick(player);
    else if (Math.random() < 0.02) { bot.state = 'block'; bot.stateTimer = 15; }
  }
}

// BỘ ĐẾM THỜI GIAN
setInterval(() => {
  if (gameRunning && timerSeconds > 0) {
    timerSeconds--;
    document.getElementById('timerVal').innerText = timerSeconds < 10 ? '0' + timerSeconds : timerSeconds;
    if (timerSeconds === 0) {
      if (p1.hp > p2.hp) triggerGameOver("PLAYER 1 THẮNG (TÍNH ĐIỂM)!");
      else if (p2.hp > p1.hp) triggerGameOver("PLAYER 2 THẮNG (TÍNH ĐIỂM)!");
      else triggerGameOver("HÒA NHAU!");
    }
  }
}, 1000);

// VÒNG LẶP CHÍNH (GAME LOOP)
function animate() {
  requestAnimationFrame(animate);

  if (gameRunning) {
    // Kiểm tra K.O
    if (p1.hp <= 0) {
      triggerGameOver("PLAYER 2 (RED) CHIẾN THẮNG!");
    } else if (p2.hp <= 0) {
      triggerGameOver("PLAYER 1 (BLUE) CHIẾN THẮNG!");
    }

    // Điều khiển Player 1
    if (keys['a']) p1.vx = -0.08;
    else if (keys['d']) p1.vx = 0.08;
    else p1.vx = 0;

    if (keys['w'] && p1.onGround) { p1.vy = 0.42; p1.onGround = false; }
    if (keys['s']) p1.state = 'crouch';
    if (keys['j']) p1.punch(p2);
    if (keys['k']) p1.kick(p2);
    if (keys['l']) { p1.state = 'block'; p1.stateTimer = 5; }

    // Điều khiển Player 2 (Người hoặc Bot)
    if (is2Player) {
      if (keys['arrowleft']) p2.vx = -0.08;
      else if (keys['arrowright']) p2.vx = 0.08;
      else p2.vx = 0;

      if (keys['arrowup'] && p2.onGround) { p2.vy = 0.42; p2.onGround = false; }
      if (keys['arrowdown']) p2.state = 'crouch';
      if (keys[',']) p2.punch(p1);
      if (keys['.']) p2.kick(p1);
      if (keys['/']) { p2.state = 'block'; p2.stateTimer = 5; }
    } else {
      updateBot(p2, p1);
    }
  }

  p1.update(p2);
  p2.update(p1);

  // Cập nhật thanh máu HUD
  document.getElementById('p1-hp').style.width = ((p1.hp / p1.maxHp) * 100) + '%';
  document.getElementById('p2-hp').style.width = ((p2.hp / p2.maxHp) * 100) + '%';

  renderer.render(scene, camera);
}

resizeBg();
animate();
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

print("🌐 [3/4] Mở đường hầm Cloudflare Tunnel công khai...")

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
print("⏳ Đang dò tìm liên kết public URL...")

for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*70)
if public_url:
    print("🚀 [4/4] DRUG FIGHTER v3 ĐÃ SẴN SÀNG VỚI KHUNG NGƯỜI PHÂN TỬ MỚI!")
    print("\n👉 BẤM VÀO ĐƯỜNG LINK DƯỚI ĐÂY ĐỂ TRẢI NGHIỆM:")
    print(f"🔗 {public_url}")
else:
    print("❌ Vui lòng kiểm tra lại log kết nối Cloudflare.")
print("="*70 + "\n")

📦 [1/4] Đang cài đặt thư viện RDKit, Flask và Cloudflare Tunnel...
🧬 [2/4] Khởi tạo Bộ xử lý Vũ khí Phân tử RDKit...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🌐 [3/4] Mở đường hầm Cloudflare Tunnel công khai...
⏳ Đang dò tìm liên kết public URL...

🚀 [4/4] DRUG FIGHTER v3 ĐÃ SẴN SÀNG VỚI KHUNG NGƯỜI PHÂN TỬ MỚI!

👉 BẤM VÀO ĐƯỜNG LINK DƯỚI ĐÂY ĐỂ TRẢI NGHIỆM:
🔗 https://baskets-measured-marriage-agencies.trycloudflare.com

